# CNN-EXP-002 — presence + intensity

Меняем только выход модели: общий backbone из CNN-EXP-001 получает две головы.
Presence предсказывает `count > 0`, intensity будет предсказывать `log1p(count)` на положительных позициях.

In [1]:
from pathlib import Path
import sys

from IPython.display import display
import torch
import torch_directml

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
    forward_both_strands_multitask,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)

Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))

DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [4]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = set(split.intervals("train")["contig"])
validation_contigs = set(split.intervals("validation")["contig"])

assert train_contigs.isdisjoint(validation_contigs)

display(split.summary)
print("FASTA contigs:", len(sequences))
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))

,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


FASTA contigs: 47
Train contigs: 9
Validation contigs: 3


In [5]:
profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

train_batch = train_generator.sample_batch(batch_size=4)

print("Обычный вход:", train_batch.x.shape)
print("Обе ориентации:", train_batch.x_both_strands.shape)
print("Target:", train_batch.target.shape)
print("Count:", train_batch.count.shape)
print("Mask:", train_batch.mask.shape)
print("Положительных:", int(train_batch.target.sum()))
print("Sampling branches:", train_batch.sampling_branch)

Обычный вход: torch.Size([4, 5, 1284])
Обе ориентации: torch.Size([8, 5, 1284])
Target: torch.Size([4, 2, 1024])
Count: torch.Size([4, 2, 1024])
Mask: torch.Size([4, 2, 1024])
Положительных: 33
Sampling branches: ('negative', 'positive', 'positive', 'negative')


## Двухголовая модель

Backbone полностью совпадает с CNN-EXP-001. Добавляется только второй выходной канал.

In [6]:
multitask_model = create_cnn_presence_intensity().to(device)
multitask_model.eval()

x_both_gpu = train_batch.x_both_strands.to(device)

with torch.no_grad():
    presence_logits, intensity_prediction = multitask_model(x_both_gpu)

parameter_count = sum(
    parameter.numel()
    for parameter in multitask_model.parameters()
    if parameter.requires_grad
)

expected_shape = (
    2 * len(train_batch.tiles),
    1,
    profile_config.input_length,
)

print("Общий вход:", x_both_gpu.shape, x_both_gpu.device)
print("Presence:", presence_logits.shape, presence_logits.device)
print("Intensity:", intensity_prediction.shape, intensity_prediction.device)
print("Параметров:", parameter_count)

assert presence_logits.shape == expected_shape
assert intensity_prediction.shape == expected_shape
assert parameter_count == 46_466

Общий вход: torch.Size([8, 5, 1284]) privateuseone:0
Presence: torch.Size([8, 1, 1284]) privateuseone:0
Intensity: torch.Size([8, 1, 1284]) privateuseone:0
Параметров: 46466


In [7]:
multitask_model.eval()

with torch.no_grad():
    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        multitask_model,
        x_both_gpu,
        batch_size=len(train_batch.tiles),
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

print("Plus presence:", plus_presence.shape)
print("Minus presence reverse:", minus_presence_reverse.shape)
print("Plus intensity:", plus_intensity.shape)
print("Minus intensity reverse:", minus_intensity_reverse.shape)

assert plus_presence.shape == (4, 1, 1024)
assert minus_presence_reverse.shape == (4, 1, 1024)
assert plus_intensity.shape == (4, 1, 1024)
assert minus_intensity_reverse.shape == (4, 1, 1024)

Plus presence: torch.Size([4, 1, 1024])
Minus presence reverse: torch.Size([4, 1, 1024])
Plus intensity: torch.Size([4, 1, 1024])
Minus intensity reverse: torch.Size([4, 1, 1024])


In [8]:
plus_count = (
    train_batch.count[:, 0:1, :]
    .contiguous()
)

minus_count_reverse = (
    train_batch.count[:, 1:2, :]
    .flip(-1)
    .contiguous()
)

plus_intensity_target = torch.log1p(
    plus_count
)

minus_intensity_target_reverse = torch.log1p(
    minus_count_reverse
)

plus_intensity_mask = (
    plus_count > 0
)

minus_intensity_mask_reverse = (
    minus_count_reverse > 0
)

print("Plus count:", plus_count.shape)
print("Minus count reverse:", minus_count_reverse.shape)

print(
    "Положительных plus:",
    int(plus_intensity_mask.sum()),
)

print(
    "Положительных minus:",
    int(minus_intensity_mask_reverse.sum()),
)

print(
    "Диапазон intensity target:",
    float(
        torch.cat(
            (
                plus_intensity_target[
                    plus_intensity_mask
                ],
                minus_intensity_target_reverse[
                    minus_intensity_mask_reverse
                ],
            )
        ).min()
    ),
    "—",
    float(
        torch.cat(
            (
                plus_intensity_target[
                    plus_intensity_mask
                ],
                minus_intensity_target_reverse[
                    minus_intensity_mask_reverse
                ],
            )
        ).max()
    ),
)

assert torch.equal(
    plus_intensity_mask.float(),
    train_batch.plus_target,
)

assert torch.equal(
    minus_intensity_mask_reverse.float(),
    train_batch.minus_target_reverse,
)

Plus count: torch.Size([4, 1, 1024])
Minus count reverse: torch.Size([4, 1, 1024])
Положительных plus: 32
Положительных minus: 1
Диапазон intensity target: 0.6931471824645996 — 4.143134593963623


In [8]:
import warnings


INTENSITY_LOSS_WEIGHT = 0.1

multitask_model.train()

optimizer = DirectMLAdam(
    multitask_model.parameters(),
    lr=1e-3,
)

# Presence targets и веса.
plus_target_gpu = (
    train_batch.plus_target.to(device)
)
minus_target_reverse_gpu = (
    train_batch.minus_target_reverse.to(device)
)

plus_presence_weight_gpu = (
    train_batch.plus_loss_weight.to(device)
)
minus_presence_weight_reverse_gpu = (
    train_batch.minus_loss_weight_reverse.to(device)
)

# Intensity targets.
plus_intensity_target_gpu = (
    plus_intensity_target.to(device)
)
minus_intensity_target_reverse_gpu = (
    minus_intensity_target_reverse.to(device)
)

# Положительные веса в BCE нормированы на 0.5.
# Умножение на 2 превращает их в условное распределение
# только среди положительных position-strand.
plus_intensity_weight_gpu = (
    2.0
    * train_batch.plus_loss_weight
    * plus_intensity_mask.float()
).to(device)

minus_intensity_weight_reverse_gpu = (
    2.0
    * train_batch.minus_loss_weight_reverse
    * minus_intensity_mask_reverse.float()
).to(device)

optimizer.zero_grad(set_to_none=True)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    (
        plus_presence,
        minus_presence_reverse,
        plus_intensity,
        minus_intensity_reverse,
    ) = forward_both_strands_multitask(
        multitask_model,
        x_both_gpu,
        batch_size=len(train_batch.tiles),
        context_flank=profile_config.context_flank,
        target_length=profile_config.target_length,
    )

    presence_loss = (
        dml_weighted_profile_bce_with_logits(
            plus_presence,
            minus_presence_reverse,
            plus_target_gpu,
            minus_target_reverse_gpu,
            plus_presence_weight_gpu,
            minus_presence_weight_reverse_gpu,
        )
    )

    intensity_loss = (
        dml_weighted_huber(
            plus_intensity,
            plus_intensity_target_gpu,
            plus_intensity_weight_gpu,
        )
        + dml_weighted_huber(
            minus_intensity_reverse,
            minus_intensity_target_reverse_gpu,
            minus_intensity_weight_reverse_gpu,
        )
    )

    total_loss = (
        presence_loss
        + INTENSITY_LOSS_WEIGHT * intensity_loss
    )

    total_loss.backward()
    optimizer.step()

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

presence_gradient = (
    multitask_model.profile_head.weight.grad[0]
    .abs()
    .sum()
    .item()
)

intensity_gradient = (
    multitask_model.profile_head.weight.grad[1]
    .abs()
    .sum()
    .item()
)

print("Presence loss:", presence_loss.item())
print("Intensity loss:", intensity_loss.item())
print("Total loss:", total_loss.item())
print("Presence gradient:", presence_gradient)
print("Intensity gradient:", intensity_gradient)
print("CPU fallback:", fallback_warnings)

assert presence_gradient > 0
assert intensity_gradient > 0
assert not fallback_warnings

Presence loss: 0.7836253643035889
Intensity loss: 1.026516079902649
Total loss: 0.8862769603729248
Presence gradient: 3.4608116149902344
Intensity gradient: 1.4364216327667236
CPU fallback: []


In [9]:
import math
import time
import warnings


PILOT_STEPS = 200
PILOT_BATCH_SIZE = 4
PILOT_SEED = 42
LOG_EVERY = 20
INTENSITY_LOSS_WEIGHT = 0.1

torch.manual_seed(PILOT_SEED)

pilot_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=PILOT_SEED,
)

pilot_model = (
    create_cnn_presence_intensity()
    .to(device)
)

pilot_optimizer = DirectMLAdam(
    pilot_model.parameters(),
    lr=1e-3,
)

pilot_model.train()

pilot_history = []
ema_presence = None
ema_intensity = None
ema_total = None
seen_position_strand = 0
started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, PILOT_STEPS + 1):
        batch = pilot_generator.sample_batch(
            batch_size=PILOT_BATCH_SIZE
        )

        # Вход обеих ориентаций уже объединён на CPU.
        x_gpu = batch.x_both_strands.to(device)

        # Presence targets.
        plus_target_gpu = (
            batch.plus_target.to(device)
        )
        minus_target_reverse_gpu = (
            batch.minus_target_reverse.to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight.to(device)
        )
        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse.to(device)
        )

        # Intensity targets готовим и разворачиваем на CPU.
        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )
        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(plus_count)
            .to(device)
        )
        minus_intensity_target_reverse_gpu = (
            torch.log1p(minus_count_reverse)
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        pilot_optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            pilot_model,
            x_gpu,
            batch_size=PILOT_BATCH_SIZE,
            context_flank=profile_config.context_flank,
            target_length=profile_config.target_length,
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT * intensity_loss
        )

        if not math.isfinite(total_loss.item()):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        pilot_optimizer.step()

        presence_value = presence_loss.item()
        intensity_value = intensity_loss.item()
        total_value = total_loss.item()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.95 * ema_presence
                + 0.05 * presence_value
            )
            ema_intensity = (
                0.95 * ema_intensity
                + 0.05 * intensity_value
            )
            ema_total = (
                0.95 * ema_total
                + 0.05 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        pilot_history.append(
            {
                "step": step,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": ema_presence,
                "ema_intensity": ema_intensity,
                "ema_total": ema_total,
            }
        )

        if step == 1 or step % LOG_EVERY == 0:
            elapsed = (
                time.perf_counter()
                - started_at
            )

            print(
                f"step={step:3d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = time.perf_counter() - started_at

print()
print("Время:", round(elapsed, 2), "сек")
print("Финальный EMA presence:", round(ema_presence, 6))
print("Финальный EMA intensity:", round(ema_intensity, 6))
print("Финальный EMA total:", round(ema_total, 6))
print("CPU fallback:", fallback_warnings)

assert not fallback_warnings

step=  1 presence=0.7394 intensity=0.8414 total=0.8236 ema=(0.7394, 0.8414, 0.8236) positions/s=183,242
step= 20 presence=0.6049 intensity=0.4744 total=0.6524 ema=(0.7055, 0.5504, 0.7606) positions/s=213,520
step= 40 presence=0.5876 intensity=0.4856 total=0.6362 ema=(0.6677, 0.4670, 0.7144) positions/s=215,050
step= 60 presence=0.5362 intensity=0.3854 total=0.5747 ema=(0.6368, 0.4155, 0.6783) positions/s=216,977
step= 80 presence=0.5786 intensity=0.3742 total=0.6161 ema=(0.5922, 0.3247, 0.6246) positions/s=218,191
step=100 presence=0.6943 intensity=1.2823 total=0.8226 ema=(0.5992, 0.3667, 0.6359) positions/s=221,028
step=120 presence=0.3870 intensity=0.1106 total=0.3981 ema=(0.5490, 0.3154, 0.5805) positions/s=220,797
step=140 presence=0.6866 intensity=0.7795 total=0.7645 ema=(0.5513, 0.2751, 0.5788) positions/s=221,438
step=160 presence=0.4077 intensity=0.1777 total=0.4255 ema=(0.5597, 0.2737, 0.5871) positions/s=222,636
step=180 presence=0.5035 intensity=0.2388 total=0.5274 ema=(0.51

In [10]:
from pathlib import Path


EXP002_SEED = 42
EXP002_STEPS = 10_000
EXP002_BATCH_SIZE = 4
EXP002_LOG_EVERY = 250
EXP002_CHECKPOINT_STEPS = {
    2_000,
    5_000,
    10_000,
}
INTENSITY_LOSS_WEIGHT = 0.1

EXP002_DIR = (
    PROJECT_ROOT
    / "artifacts/experiments/CNN_EXP002"
    / "presence_intensity_v1"
)

EXP002_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.manual_seed(EXP002_SEED)

exp002_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=EXP002_SEED,
)

exp002_model = (
    create_cnn_presence_intensity()
    .to(device)
)

exp002_optimizer = DirectMLAdam(
    exp002_model.parameters(),
    lr=1e-3,
)

print("Артефакты:", EXP002_DIR)
print("Параметров:", sum(
    parameter.numel()
    for parameter in exp002_model.parameters()
    if parameter.requires_grad
))
print("Шагов:", EXP002_STEPS)
print("Intensity weight:", INTENSITY_LOSS_WEIGHT)

Артефакты: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP002\presence_intensity_v1
Параметров: 46466
Шагов: 10000
Intensity weight: 0.1


In [11]:
import copy
import math
import time
import warnings

import pandas as pd


def model_state_on_cpu(model):
    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


def tree_to_cpu(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()

    if isinstance(value, dict):
        return {
            key: tree_to_cpu(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            tree_to_cpu(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            tree_to_cpu(item)
            for item in value
        )

    return value


def save_exp002_checkpoint(
    path,
    *,
    step,
    elapsed_seconds,
    seen_position_strand,
    history,
):
    checkpoint = {
        "experiment": "CNN-EXP-002",
        "model_type": "presence_intensity",
        "step": step,
        "model_state_dict": model_state_on_cpu(
            exp002_model
        ),
        "optimizer_state_dict": tree_to_cpu(
            exp002_optimizer.state_dict()
        ),
        "generator_rng_state": copy.deepcopy(
            exp002_generator.rng.bit_generator.state
        ),
        "torch_rng_state": torch.get_rng_state(),
        "profile_config": {
            "target_length": (
                profile_config.target_length
            ),
            "context_flank": (
                profile_config.context_flank
            ),
            "positive_branch_fraction": (
                profile_config
                .positive_branch_fraction
            ),
        },
        "training_config": {
            "seed": EXP002_SEED,
            "steps": EXP002_STEPS,
            "batch_size": EXP002_BATCH_SIZE,
            "learning_rate": 1e-3,
            "intensity_loss_weight": (
                INTENSITY_LOSS_WEIGHT
            ),
            "huber_delta": 1.0,
        },
        "seen_position_strand": (
            seen_position_strand
        ),
        "elapsed_seconds": elapsed_seconds,
        "history": list(history),
    }

    torch.save(checkpoint, path)


exp002_model.train()

exp002_history = []
ema_presence = None
ema_intensity = None
ema_total = None
seen_position_strand = 0
started_at = time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for step in range(1, EXP002_STEPS + 1):
        batch = exp002_generator.sample_batch(
            batch_size=EXP002_BATCH_SIZE
        )

        x_gpu = batch.x_both_strands.to(device)

        plus_target_gpu = (
            batch.plus_target.to(device)
        )
        minus_target_reverse_gpu = (
            batch.minus_target_reverse.to(device)
        )

        plus_presence_weight_gpu = (
            batch.plus_loss_weight.to(device)
        )
        minus_presence_weight_reverse_gpu = (
            batch.minus_loss_weight_reverse.to(device)
        )

        # Всё выравнивание minus выполняется на CPU.
        plus_count = (
            batch.count[:, 0:1, :]
            .contiguous()
        )
        minus_count_reverse = (
            batch.count[:, 1:2, :]
            .flip(-1)
            .contiguous()
        )

        plus_intensity_target_gpu = (
            torch.log1p(plus_count)
            .to(device)
        )
        minus_intensity_target_reverse_gpu = (
            torch.log1p(minus_count_reverse)
            .to(device)
        )

        plus_intensity_weight_gpu = (
            2.0
            * batch.plus_loss_weight
            * (plus_count > 0).float()
        ).to(device)

        minus_intensity_weight_reverse_gpu = (
            2.0
            * batch.minus_loss_weight_reverse
            * (minus_count_reverse > 0).float()
        ).to(device)

        exp002_optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_presence,
            minus_presence_reverse,
            plus_intensity,
            minus_intensity_reverse,
        ) = forward_both_strands_multitask(
            exp002_model,
            x_gpu,
            batch_size=EXP002_BATCH_SIZE,
            context_flank=(
                profile_config.context_flank
            ),
            target_length=(
                profile_config.target_length
            ),
        )

        presence_loss = (
            dml_weighted_profile_bce_with_logits(
                plus_presence,
                minus_presence_reverse,
                plus_target_gpu,
                minus_target_reverse_gpu,
                plus_presence_weight_gpu,
                minus_presence_weight_reverse_gpu,
            )
        )

        intensity_loss = (
            dml_weighted_huber(
                plus_intensity,
                plus_intensity_target_gpu,
                plus_intensity_weight_gpu,
                delta=1.0,
            )
            + dml_weighted_huber(
                minus_intensity_reverse,
                minus_intensity_target_reverse_gpu,
                minus_intensity_weight_reverse_gpu,
                delta=1.0,
            )
        )

        total_loss = (
            presence_loss
            + INTENSITY_LOSS_WEIGHT
            * intensity_loss
        )

        presence_value = presence_loss.item()
        intensity_value = intensity_loss.item()
        total_value = total_loss.item()

        if not all(
            math.isfinite(value)
            for value in (
                presence_value,
                intensity_value,
                total_value,
            )
        ):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        exp002_optimizer.step()

        if ema_total is None:
            ema_presence = presence_value
            ema_intensity = intensity_value
            ema_total = total_value
        else:
            ema_presence = (
                0.99 * ema_presence
                + 0.01 * presence_value
            )
            ema_intensity = (
                0.99 * ema_intensity
                + 0.01 * intensity_value
            )
            ema_total = (
                0.99 * ema_total
                + 0.01 * total_value
            )

        seen_position_strand += int(
            batch.mask.sum()
        )

        if (
            step == 1
            or step % EXP002_LOG_EVERY == 0
        ):
            elapsed = (
                time.perf_counter()
                - started_at
            )

            record = {
                "step": step,
                "presence_loss": presence_value,
                "intensity_loss": intensity_value,
                "total_loss": total_value,
                "ema_presence": ema_presence,
                "ema_intensity": ema_intensity,
                "ema_total": ema_total,
                "elapsed_seconds": elapsed,
                "seen_position_strand": (
                    seen_position_strand
                ),
                "positions_per_second": (
                    seen_position_strand / elapsed
                ),
            }

            exp002_history.append(record)

            print(
                f"step={step:5d} "
                f"presence={presence_value:.4f} "
                f"intensity={intensity_value:.4f} "
                f"total={total_value:.4f} "
                f"ema=({ema_presence:.4f}, "
                f"{ema_intensity:.4f}, "
                f"{ema_total:.4f}) "
                f"positions/s="
                f"{seen_position_strand / elapsed:,.0f}"
            )

        if step in EXP002_CHECKPOINT_STEPS:
            checkpoint_path = (
                EXP002_DIR
                / f"model_step_{step:05d}.pt"
            )

            save_exp002_checkpoint(
                checkpoint_path,
                step=step,
                elapsed_seconds=(
                    time.perf_counter()
                    - started_at
                ),
                seen_position_strand=(
                    seen_position_strand
                ),
                history=exp002_history,
            )

fallback_warnings = [
    str(item.message)
    for item in caught
    if "fall back" in str(item.message).lower()
]

elapsed = time.perf_counter() - started_at

final_checkpoint_path = (
    EXP002_DIR / "exp002_final.pt"
)
history_path = (
    EXP002_DIR / "training_history.csv"
)

save_exp002_checkpoint(
    final_checkpoint_path,
    step=EXP002_STEPS,
    elapsed_seconds=elapsed,
    seen_position_strand=seen_position_strand,
    history=exp002_history,
)

pd.DataFrame(exp002_history).to_csv(
    history_path,
    index=False,
)

print()
print("Общее время:", round(elapsed, 2), "сек")
print(
    "Время шага:",
    round(elapsed / EXP002_STEPS, 4),
    "сек",
)
print(
    "Увидено position-strand:",
    seen_position_strand,
)
print(
    "Финальный EMA presence:",
    round(ema_presence, 6),
)
print(
    "Финальный EMA intensity:",
    round(ema_intensity, 6),
)
print(
    "Финальный EMA total:",
    round(ema_total, 6),
)
print("CPU fallback:", fallback_warnings)
print("Checkpoint:", final_checkpoint_path)
print("История:", history_path)

assert not fallback_warnings
assert final_checkpoint_path.is_file()
assert history_path.is_file()

step=    1 presence=0.7394 intensity=0.8414 total=0.8236 ema=(0.7394, 0.8414, 0.8236) positions/s=161,432
step=  250 presence=0.4622 intensity=0.4512 total=0.5073 ema=(0.5593, 0.3498, 0.5943) positions/s=219,700
step=  500 presence=0.5655 intensity=0.0187 total=0.5674 ema=(0.5075, 0.2438, 0.5318) positions/s=222,146
step=  750 presence=0.5208 intensity=0.2626 total=0.5470 ema=(0.5039, 0.2562, 0.5295) positions/s=214,794
step= 1000 presence=0.2950 intensity=0.2110 total=0.3161 ema=(0.4922, 0.2534, 0.5175) positions/s=212,190
step= 1250 presence=0.3089 intensity=0.9258 total=0.4015 ema=(0.4921, 0.2493, 0.5170) positions/s=213,989
step= 1500 presence=0.4103 intensity=0.1809 total=0.4284 ema=(0.4751, 0.2266, 0.4978) positions/s=214,814
step= 1750 presence=0.5325 intensity=0.0858 total=0.5411 ema=(0.4667, 0.2573, 0.4924) positions/s=215,365
step= 2000 presence=0.3398 intensity=0.3344 total=0.3733 ema=(0.4638, 0.2677, 0.4906) positions/s=215,811
step= 2250 presence=0.4629 intensity=0.0318 to

In [9]:
from ml_project.epic_cnn.evaluation import (
    evaluate_presence_intensity_model,
    load_cnn_presence_intensity_checkpoint,
)


EXP002_CHECKPOINT_PATH = (
    PROJECT_ROOT
    / "artifacts/experiments/CNN_EXP002"
    / "presence_intensity_v1"
    / "exp002_final.pt"
)

loaded_exp002 = (
    load_cnn_presence_intensity_checkpoint(
        EXP002_CHECKPOINT_PATH,
        device=device,
    )
)

validation_model = loaded_exp002.model
validation_config = loaded_exp002.profile_config
checkpoint = loaded_exp002.checkpoint

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=validation_config,
    seed=42,
)

print("Checkpoint step:", checkpoint["step"])
print("Validation tiles:", len(validation_generator.tile_index))
print(
    "Validation position-strand:",
    int(
        validation_generator
        .tile_index["allowed_position_strand"]
        .sum()
    ),
)
print(
    "Validation positives:",
    validation_generator.total_positive,
)
print(
    "Device:",
    next(validation_model.parameters()).device,
)

Checkpoint step: 10000
Validation tiles: 47743
Validation position-strand: 83807486
Validation positives: 56205
Device: privateuseone:0


In [10]:
validation_result = (
    evaluate_presence_intensity_model(
        validation_model,
        validation_generator,
        device=device,
        batch_size=8,
        score_decimals=5,
        progress_every_batches=500,
    )
)

tiles=     8/47743 positions=    11484 positions/s=1,150,851
tiles=  4000/47743 positions=  6861742 positions/s=2,264,212
tiles=  8000/47743 positions= 14166170 positions/s=2,269,953
tiles= 12000/47743 positions= 21458144 positions/s=2,248,138
tiles= 16000/47743 positions= 28627148 positions/s=2,279,118
tiles= 20000/47743 positions= 35584246 positions/s=2,262,741
tiles= 24000/47743 positions= 42571942 positions/s=2,278,817
tiles= 28000/47743 positions= 49714988 positions/s=2,295,543
tiles= 32000/47743 positions= 56582858 positions/s=2,302,726
tiles= 36000/47743 positions= 63201944 positions/s=2,251,916
tiles= 40000/47743 positions= 70164960 positions/s=2,229,203
tiles= 44000/47743 positions= 77380732 positions/s=2,245,782
tiles= 47743/47743 positions= 83807486 positions/s=2,233,617


In [11]:
VALIDATION_METRICS_PATH = (
    EXP002_CHECKPOINT_PATH.parent
    / "validation_metrics.csv"
)

validation_result.metrics.to_csv(
    VALIDATION_METRICS_PATH,
    index=False,
)

overall_metrics = (
    validation_result.metrics
    .loc[
        validation_result.metrics["segment"]
        == "overall"
    ]
    .reset_index(drop=True)
)

display(overall_metrics)

print(
    "Validation time:",
    round(validation_result.elapsed_seconds, 2),
    "sec",
)
print(
    "Position-strand/s:",
    f"{validation_result.positions_per_second:,.0f}",
)
print(
    "CPU fallback:",
    list(validation_result.fallback_warnings),
)
print("Metrics:", VALIDATION_METRICS_PATH)

,score,segment,position_strand,positives,prevalence,average_precision,ap_lift,epic_spearman,quantized_spearman
0,presence_probability,overall,83807486,56205,0.000671,0.013621,20.309587,0.159182,0.149752
1,intensity_prediction,overall,83807486,56205,0.000671,0.011986,17.872652,0.177420,0.190550
2,expected_signal,overall,83807486,56205,0.000671,0.014379,21.441256,0.174195,0.167661


Validation time: 37.63 sec
Position-strand/s: 2,226,947
CPU fallback: []
Metrics: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP002\presence_intensity_v1\validation_metrics.csv


In [12]:
per_contig_metrics = (
    validation_result.metrics
    .loc[
        validation_result.metrics["segment"]
        != "overall"
    ]
    .sort_values(
        ["segment", "score"]
    )
    .reset_index(drop=True)
)

display(per_contig_metrics)

,score,segment,position_strand,positives,prevalence,average_precision,ap_lift,epic_spearman,quantized_spearman
0,expected_signal,NC_064034.1,33949114,23529,0.000693,0.016163,23.320450,0.183534,0.180779
1,intensity_prediction,NC_064034.1,33949114,23529,0.000693,0.013618,19.648547,0.186134,0.192536
2,presence_probability,NC_064034.1,33949114,23529,0.000693,0.015113,21.805833,0.170454,0.166230
3,expected_signal,NC_064041.1,24146754,13984,0.000579,0.012891,22.259376,0.214187,0.211786
4,intensity_prediction,NC_064041.1,24146754,13984,0.000579,0.010820,18.683723,0.216798,0.222645
5,presence_probability,NC_064041.1,24146754,13984,0.000579,0.011855,20.470664,0.197965,0.194199
6,expected_signal,NC_064042.1,25711618,18692,0.000727,0.013813,19.000989,0.208917,0.206783
7,intensity_prediction,NC_064042.1,25711618,18692,0.000727,0.011320,15.571006,0.212086,0.219576
8,presence_probability,NC_064042.1,25711618,18692,0.000727,0.013570,18.666241,0.187739,0.184085
